# Geolokasi Obstacle Trotoar dari Foto HP (Metric Monocular Depth)

Input: foto trotoar dari HP yang EXIF-nya berisi GPS dan arah kamera (diuji dengan OPPO A7, 4160×3120). Kamera lain (Samsung, dll.) didukung: focal length ditentukan **per kamera** dari EXIF masing-masing.

**Alur per foto:**
1. Baca pose dari EXIF (GPS, heading magnetik → heading geografis, kamera)
2. Tentukan focal length per kamera (kalibrasi → EXIF → median estimasi Depth Pro)
3. SegFormer → mask trotoar
4. Depth Pro → depth dalam meter
5. Back-project piksel trotoar ke 3D → fit bidang tanah (RANSAC) → pitch/roll + tinggi kamera hasil fit
6. Koreksi skala depth dengan tinggi kamera yang diukur (`CAM_HEIGHT_M`)
7. YOLO → bounding box obstacle → titik kontak (tepi bawah box)
8. Ray titik kontak ∩ bidang tanah → posisi 3D di frame kamera → ENU (heading) → lat/lon

**Struktur:** Bagian 1–3 konfigurasi dan fungsi · Bagian 4 load model dan focal per kamera · Bagian 5 debug satu foto · Bagian 6 batch dan export Shapefile · Bagian 7 cara cek di QGIS.

**Perubahan berdasarkan hasil uji awal (3 foto, Semarang, Sept 2025)** ditandai *"diubah dari hasil uji"* di tabel parameter. Parameter yang hasilnya sudah memadai dibiarkan seperti versi awal.

## 0. Setup (Colab)

In [ ]:
# KENAPA transformers==4.57.6:
#   checkpoint SegFormer dilatih dengan transformers 4.x. Di transformers 5.x nama layer SegFormer diganti
#   (mis. encoder.patch_embeddings.0 -> stages.0.patch_embeddings, attention.self.query -> attention.q_proj),
#   sehingga checkpoint gagal di-load. Depth Pro butuh >= 4.50, jadi dipakai rilis 4.x terakhir.
# Setelah install: Runtime > Restart session, lalu lanjut dari Bagian 1 (jangan jalankan sel ini lagi).
!pip install -q "transformers==4.57.6" ultralytics exifread geopandas pytorch-lightning pygeomag huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Konfigurasi

### Dasar pemilihan parameter

Jenis: **ukuran lapangan** = diukur saat pemotretan · **turunan** = dihitung dari rumus/data · **heuristik** = nilai praktis yang perlu divalidasi.
Hitungan contoh memakai h = 1.4 m dan f ≈ 3018 px (OPPO A7, lebar 4160 px).

| Parameter | Nilai | Jenis | Status | Dasar |
|---|---|---|---|---|
| `DEPTH_BACKEND` | `depthpro` | pilihan model | awal | Model metric depth yang ikut mengestimasi focal length dari satu gambar (Bochkovskii et al., 2024). Dibutuhkan karena tidak semua HP menulis focal yang valid di EXIF. |
| `CAM_HEIGHT_M` | 1.4 m | ukuran lapangan | diubah (1.2 → 1.4) | Tinggi lensa HP di atas permukaan trotoar. Acuan skala metrik: `D = h·f / (v − v_horizon)`, jadi semua jarak sebanding linear dengan h; error 10 cm ≈ 7% error jarak. **Uji awal:** skala Depth Pro antar foto berkisar 1.05–1.84×, jadi koreksi pakai tinggi kamera wajib. |
| `MAG_DECLINATION_DEG` | `auto` | turunan model | awal | Heading EXIF relatif utara magnetik (`GPSImgDirectionRef = M`). Deklinasi dihitung dari World Magnetic Model (WMM2025) per posisi dan tanggal foto; Semarang, Sept 2025 ≈ 0.75° timur. |
| `CAMERA_CALIB` | `{}` | ukuran lapangan (opsional) | baru | Focal hasil kalibrasi checkerboard per model kamera. Sumber paling akurat, diprioritaskan di atas EXIF. |
| `SENSOR_WIDTH_DB` | `{}` | spesifikasi (opsional) | baru | Lebar sensor per model kalau tidak ada di EXIF, dipakai bersama `FocalLength` (mm). |
| Focal per kamera | otomatis | turunan EXIF/data | diubah dari hasil uji | Urutan: kalibrasi → EXIF 35mm (≥ 10 mm, `f = f35 × diagonal_px / 43.27`) → focal mm + lebar sensor → **median** estimasi Depth Pro dari semua foto kamera yang sama. **Uji awal:** estimasi focal Depth Pro per foto dari HP yang sama tidak konsisten (2877 vs 3018 px), jadi focal dikunci per kamera. Satu kamera = Make + Model + focal mm + ukuran gambar. |
| `PLANE_MIN_Z` | 1.0 m | batas aman | awal | Dengan VFOV ≈ 55° dan pitch ≈ −2°, tanah terdekat yang terlihat ≈ 1.4 / tan(29°) ≈ 2.5 m, jadi batas ini praktis tidak memotong data; hanya membuang depth yang tidak wajar. |
| `PLANE_MAX_Z` | 15 m | heuristik | awal | Asumsi trotoar datar hanya berlaku lokal; makin jauh sudut pandang ke tanah makin landai dan error absolut depth membesar. **Uji awal:** fit bidang stabil (pitch_fit −1.6° s/d −2.1°). |
| `MAX_DIST_M` | 15 m | turunan rumus | diubah (20 → 15) | Sensitivitas jarak terhadap error piksel tepi bawah box: `|dD/dv| = D² / (h·f)`. Pada h = 1.4 m: 10 m → 0.024 m/px, 15 m → 0.053 m/px, 20 m → 0.095 m/px. **Uji awal:** satu-satunya outlier (`diff_m` 3.25 m) adalah obstacle pada 18.5 m. |
| `MIN_CONF` | 0.5 | sementara | diubah (0.35 → 0.5) | Default `predict` Ultralytics 0.25. **Uji awal:** outlier tadi ber-confidence 0.47. Nilai ideal: confidence dengan F1 maksimum di `F1_curve.png` hasil training YOLO (`result_v1/`). |
| `USE_DEPTH_POSITION` | `False` | pilihan metode | awal | Posisi dari ray titik kontak ∩ bidang hasil fit ribuan piksel. Depth langsung di titik kontak lebih noisy (batas objek–tanah sering blur di depth map), jadi hanya dipakai sebagai pembanding (`diff_m`). **Uji awal:** 23 dari 24 obstacle `diff_m` < 0.35 m. |

**Konstanta di dalam fungsi:**

| Konstanta | Nilai | Jenis | Dasar |
|---|---|---|---|
| RANSAC `iters` | 300 | turunan rumus | `N = log(1−p) / log(1−w³)` (Fischler & Bolles, 1981). p = 0.999, rasio inlier w = 0.3 → N ≈ 252. |
| RANSAC `thr` | 0.08 m | heuristik | Toleransi jarak titik ke bidang (kekasaran ubin + noise depth). Cek rasio inlier di sel 5.4. |
| RANSAC `min_inliers` | 300 | heuristik | Minimal titik untuk fit stabil; di bawahnya fallback ke kamera datar. |
| Toleransi normal tanah | 35° | heuristik dengan acuan | Ramp aksesibilitas maks 1:12 ≈ 4.8° (ADA 405.2) dan HP genggam jarang miring > 20°; 35° tetap menolak dinding (≈ 90°). |
| `bottom_on_sidewalk` | strip 6 px, ≥ 30% | heuristik | Obstacle dianggap di trotoar kalau sebagian tepi bawah box menyentuh mask; toleran terhadap mask yang tidak rapi di batas objek. |
| Ambang EXIF 35mm valid | ≥ 10 mm | heuristik dengan acuan | Lensa HP utama setara ~13–30 mm; OPPO A7 menulis 3 (bug firmware). |

Referensi: Bochkovskii et al. (2024) https://arxiv.org/abs/2410.02073 · Fischler & Bolles (1981), *Comm. ACM* 24(6) · WMM https://www.ncei.noaa.gov/products/world-magnetic-model · pyGeoMag https://pygeomag.readthedocs.io/ · Ultralytics predict https://docs.ultralytics.com/modes/predict/ · ADA Standards https://www.access-board.gov/ada/

In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download

# --- Foto input -----------------------------------------------------------------
# Pilihan A: foto sendiri di Google Drive (perlu sel mount Drive di Bagian 0)
IMAGE_FOLDER = Path("/content/drive/MyDrive/sidewalk_photos/")
# Pilihan B: foto contoh dari repo (sudah dianonimkan, EXIF utuh)
# !git clone -q https://github.com/wahyunanandika/sidewalk-obstacle-geolocation.git
# IMAGE_FOLDER = Path("/content/sidewalk-obstacle-geolocation/sample_data/photos/")

# --- Bobot model dari Hugging Face (isi file sama dengan yang dipakai saat pengujian) ---
HF_REPO        = "wahyunan/sidewalk-obstacle-geolocation"
SEGFORMER_CKPT = hf_hub_download(HF_REPO, "segformer_final.ckpt")         # SegFormer-B0, checkpoint epoch terakhir
YOLO_WEIGHTS   = hf_hub_download(HF_REPO, "yolo-obstacle-detection.pt")   # YOLO11s, 13 kelas (best.pt)
# Alternatif, kalau bobot ada di Drive sendiri:
# SEGFORMER_CKPT = "/content/drive/MyDrive/model_finetune_segformer/modelv1/segformer_final.ckpt"
# YOLO_WEIGHTS   = "/content/drive/MyDrive/dataset_dilans_v8/result_v1/weights/best.pt"
ID2LABEL       = {0: "background", 1: "sidewalk"}

cfg = dict(
    DEPTH_BACKEND="depthpro",    # ikut mengestimasi focal -> dibutuhkan untuk HP yang EXIF focal-nya tidak valid
    CAM_HEIGHT_M=1.4,            # UKUR tinggi lensa HP di atas trotoar; acuan skala (D ∝ h)
    MAG_DECLINATION_DEG="auto",  # WMM2025 dari GPS + tanggal EXIF; atau isi angka tetap (derajat, timur +)
    CAMERA_CALIB={},             # {"OPPO OPPO A7": (f_px, lebar_px_saat_kalibrasi)} -> prioritas tertinggi
    SENSOR_WIDTH_DB={},          # {"OPPO OPPO A7": 4.8} lebar sensor (mm) kalau tidak ada di EXIF
    PLANE_MIN_Z=1.0,             # batas aman; tanah terdekat terlihat ~2.5 m
    PLANE_MAX_Z=15.0,            # trotoar dianggap datar hanya secara lokal
    MAX_DIST_M=15.0,             # diubah dari 20: |dD/dv| = D²/(h·f) ~0.05 m/px di 15 m; outlier uji awal di 18.5 m
    MIN_CONF=0.5,                # diubah dari 0.35: outlier uji awal conf 0.47; idealnya conf F1-maks dari F1_curve.png
    USE_DEPTH_POSITION=False,    # posisi dari ray ∩ bidang; depth langsung hanya pembanding (diff_m)
)

## 2. Import

In [ ]:
import math, re
import datetime as _dt
from functools import lru_cache
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import pytorch_lightning as pl
import exifread
import geopandas as gpd
from shapely.geometry import Point
from PIL import Image, ImageOps
from pygeomag import GeoMag
from transformers import (SegformerForSemanticSegmentation, SegformerImageProcessor,
                          AutoImageProcessor, AutoModelForDepthEstimation)
from ultralytics import YOLO

import transformers
print("transformers", transformers.__version__)   # harus 4.57.6; kalau 5.x, ulangi Bagian 0 lalu restart session
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 3. Fungsi-fungsi

### 3.1 Wrapper SegFormer
Class ini harus sama dengan saat training supaya checkpoint Lightning bisa di-load.

In [ ]:
class SegformerFinetuner(pl.LightningModule):
    def __init__(self, id2label, train_dataloader=None, val_dataloader=None,
                 test_dataloader=None, metrics_interval=100):
        super().__init__()
        self.id2label = id2label
        self.num_classes = len(id2label)
        self.label2id = {v: k for k, v in id2label.items()}
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/segformer-b0-finetuned-ade-512-512",
            return_dict=True, num_labels=self.num_classes,
            id2label=self.id2label, label2id=self.label2id,
            ignore_mismatched_sizes=True,   # kepala 150 kelas ADE20K diganti 2 kelas; bobotnya dari checkpoint
        )

    def forward(self, images, masks=None):
        return self.model(pixel_values=images, labels=masks)

### 3.2 Membaca EXIF dan heading geografis

| Tag | Contoh (OPPO A7) | Catatan |
|---|---|---|
| `GPSLatitude/Longitude` | −6.9782, 110.4107 | posisi HP |
| `GPSImgDirection` | 124.72 | arah kamera, searah jarum jam |
| `GPSImgDirectionRef` | `M` | **utara magnetik** → perlu koreksi deklinasi (`T` = utara geografis) |
| `Make` / `Model` | OPPO / OPPO A7 | identitas kamera untuk focal per kamera |
| `FocalLength` | 3.46 mm | focal fisik; perlu lebar sensor untuk dikonversi ke piksel |
| `FocalLengthIn35mmFilm` | 3 | **tidak valid** di OPPO A7 (bug firmware), diabaikan otomatis |

- Foto tanpa GPS atau heading **di-skip**, tidak di-default ke 0. Default 0 artinya semua foto dianggap menghadap utara, yang merupakan salah satu penyebab titik "melompat" di kode lama.
- **Deklinasi** dihitung offline dengan World Magnetic Model (koefisien dibundel di `pygeomag`). WMM2025 berlaku 2025–2030; foto lebih lama memakai WMM2020/WMM2015v2.

In [ ]:
def _r2f(r):
    try:
        return float(r.num) / float(r.den)
    except Exception:
        try:
            return float(r[0]) / float(r[1])
        except Exception:
            return float(r)


def read_exif(path: str) -> dict:
    with open(path, "rb") as f:
        tags = exifread.process_file(f, details=False)
    out = dict(lat=None, lon=None, alt=None, heading=None, heading_ref=None, pitch=None, roll=None,
               focal_mm=None, focal35=None, datetime=None, make=None, model=None, sensor_width_mm=None)

    if all(k in tags for k in ["GPS GPSLatitude", "GPS GPSLongitude",
                               "GPS GPSLatitudeRef", "GPS GPSLongitudeRef"]):
        la, lo = tags["GPS GPSLatitude"].values, tags["GPS GPSLongitude"].values
        lat = _r2f(la[0]) + _r2f(la[1]) / 60 + _r2f(la[2]) / 3600
        lon = _r2f(lo[0]) + _r2f(lo[1]) / 60 + _r2f(lo[2]) / 3600
        if str(tags["GPS GPSLatitudeRef"]).upper().startswith("S"):
            lat = -lat
        if str(tags["GPS GPSLongitudeRef"]).upper().startswith("W"):
            lon = -lon
        out["lat"], out["lon"] = lat, lon
    if "GPS GPSAltitude" in tags:
        out["alt"] = _r2f(tags["GPS GPSAltitude"].values[0])
    if "GPS GPSImgDirection" in tags:
        out["heading"] = _r2f(tags["GPS GPSImgDirection"].values[0])
        out["heading_ref"] = str(tags.get("GPS GPSImgDirectionRef", "")).strip().upper() or None
    if "EXIF DateTimeOriginal" in tags:
        out["datetime"] = str(tags["EXIF DateTimeOriginal"])            # "YYYY:MM:DD HH:MM:SS"
    elif "GPS GPSDate" in tags:
        out["datetime"] = str(tags["GPS GPSDate"]) + " 12:00:00"
    if "EXIF FocalLength" in tags:
        out["focal_mm"] = _r2f(tags["EXIF FocalLength"].values[0])
    if "EXIF FocalLengthIn35mmFilm" in tags:
        v = _r2f(tags["EXIF FocalLengthIn35mmFilm"].values[0])
        out["focal35"] = v if v > 0 else None

    # --- identitas kamera (untuk focal per kamera) ---
    if "Image Make" in tags:
        out["make"] = str(tags["Image Make"]).strip()
    if "Image Model" in tags:
        out["model"] = str(tags["Image Model"]).strip()
    # lebar sensor dari FocalPlaneXResolution (piksel per satuan di bidang sensor), kalau kamera menulisnya
    if "EXIF FocalPlaneXResolution" in tags and "EXIF ExifImageWidth" in tags:
        unit = tags.get("EXIF FocalPlaneResolutionUnit")
        unit_mm = {2: 25.4, 3: 10.0, 4: 1.0}.get(int(unit.values[0]) if unit else 2, 25.4)
        xres = _r2f(tags["EXIF FocalPlaneXResolution"].values[0])
        if xres > 0:
            out["sensor_width_mm"] = float(str(tags["EXIF ExifImageWidth"])) / xres * unit_mm

    # pitch/roll/heading custom di field komentar (format kode lama), kalau ada
    for key in ["Image ImageDescription", "Image UserComment", "EXIF UserComment"]:
        if key in tags:
            kv = {k.lower(): float(v) for k, v in
                  re.findall(r"(pitch|roll|yaw|heading)\s*[:=]\s*([-\d.]+)",
                             str(tags[key]), flags=re.I)}
            out["pitch"] = kv.get("pitch", out["pitch"])
            out["roll"] = kv.get("roll", out["roll"])
            if out["heading"] is None:
                out["heading"] = kv.get("heading", kv.get("yaw"))
            if kv:
                break
    return out


@lru_cache(maxsize=None)
def _geomag(cof):
    return GeoMag(coefficients_file=cof)


def decimal_year(exif_datetime: Optional[str]) -> float:
    try:
        d = _dt.datetime.strptime(exif_datetime.strip(), "%Y:%m:%d %H:%M:%S")
    except Exception:
        d = _dt.datetime.now()
        print("  [warn] no EXIF date, using today for declination")
    days = 366 if (d.year % 4 == 0 and d.year % 100 != 0) or d.year % 400 == 0 else 365
    return d.year + (d.timetuple().tm_yday - 1 + d.hour / 24) / days


def magnetic_declination(lat, lon, exif_datetime, alt_km=0.0) -> float:
    # World Magnetic Model, offline. Timur positif, derajat.
    yr = decimal_year(exif_datetime)
    cof = ("wmm/WMM_2025.COF" if yr >= 2025 else
           "wmm/WMM_2020.COF" if yr >= 2020 else "wmm/WMM_2015v2.COF")
    return _geomag(cof).calculate(glat=lat, glon=lon, alt=alt_km, time=yr).d


def true_heading(pose: dict, declination_cfg) -> tuple:
    # Heading magnetik -> geografis kalau EXIF Ref = 'M'. Return (heading_true, deklinasi_dipakai)
    if pose["heading"] is None:
        return None, None
    h, decl = pose["heading"], 0.0
    if pose.get("heading_ref") == "M":
        decl = (magnetic_declination(pose["lat"], pose["lon"], pose.get("datetime"))
                if declination_cfg == "auto" else float(declination_cfg))
        h += decl
    return h % 360.0, decl

### 3.3 Focal length per kamera

**Kenapa per kamera, bukan per foto atau angka tetap:** focal adalah sifat lensa, jadi semua foto dari kamera dan lensa yang sama harus memakai focal yang sama. Angka tetap tidak bisa dipakai untuk HP lain, dan estimasi per foto tidak konsisten (uji awal: 2877 vs 3018 px dari HP yang sama).

Satu kamera = **Make + Model + focal mm + ukuran gambar**. Lensa ultrawide/tele di HP yang sama, atau mode crop, otomatis dianggap kamera berbeda.

Urutan sumber focal:
1. **Kalibrasi** (`CAMERA_CALIB`), diskalakan ke resolusi foto.
2. **EXIF `FocalLengthIn35mmFilm`** kalau ≥ 10 mm: `f_px = f35 × √(W² + H²) / 43.27`. Angka 43.27 mm adalah diagonal frame 35mm (36 × 24 mm), dan crop factor didefinisikan lewat diagonal, sehingga rumus ini benar untuk rasio 4:3 maupun 16:9.
3. **EXIF `FocalLength` (mm) × W / lebar sensor (mm)**, kalau lebar sensor diketahui (dari EXIF atau `SENSOR_WIDTH_DB`).
4. **Median estimasi Depth Pro** dari semua foto kamera itu. Median dipakai karena tahan terhadap satu-dua estimasi yang meleset.

Cek fisik untuk OPPO A7: 3.46 mm × 4160 / 3018 ≈ 4.8 mm lebar sensor, sesuai ukuran umum sensor 13 MP kelas 1/3".

In [ ]:
def camera_key(pose: dict, W: int, H: int) -> str:
    fmm = f"{pose['focal_mm']:.2f}mm" if pose.get("focal_mm") else "nofocal"
    return f"{pose.get('make') or '?'} {pose.get('model') or '?'} | {fmm} | {W}x{H}"


def focal_from_exif(W: int, H: int, pose: dict, cfg: dict):
    # Focal (px) dari kalibrasi atau EXIF. Return (f_px, sumber) atau (None, None).
    model = f"{pose.get('make') or ''} {pose.get('model') or ''}".strip()

    cal = cfg.get("CAMERA_CALIB", {}).get(model)          # 1. kalibrasi: (f_px, lebar_kalibrasi)
    if cal:
        f_cal, w_cal = cal
        return f_cal * W / w_cal, "calibration"

    f35 = pose.get("focal35")                             # 2. 35mm-equivalent, berbasis diagonal
    if f35 and f35 >= 10:                                 #    < 10 mm tidak masuk akal untuk lensa HP
        return f35 * (W ** 2 + H ** 2) ** 0.5 / 43.27, "exif-35mm"

    fmm = pose.get("focal_mm")                            # 3. focal mm + lebar sensor
    sw = pose.get("sensor_width_mm") or cfg.get("SENSOR_WIDTH_DB", {}).get(model)
    if fmm and sw:
        return fmm / sw * W, "exif-mm+sensor"

    return None, None


def estimate_camera_focals(image_list, depth_model, cfg, cache_csv=None):
    # {camera_key: (f_px, sumber)}. Kamera tanpa focal EXIF -> median estimasi Depth Pro semua fotonya.
    # Hasil estimasi per foto disimpan di cache_csv supaya run berikutnya tidak menghitung ulang.
    cache = {}
    if cache_csv is not None and Path(cache_csv).exists():
        cache = pd.read_csv(cache_csv).set_index("image")["f_pred"].to_dict()

    rows = []
    for p in image_list:
        pose = read_exif(str(p))
        with Image.open(p) as im:
            W, H = ImageOps.exif_transpose(im).size
        key = camera_key(pose, W, H)
        f, src = focal_from_exif(W, H, pose, cfg)
        f_pred = None
        if f is None:
            f_pred = cache.get(p.name)
            if f_pred is None or pd.isna(f_pred):
                img = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
                _, f_pred = depth_model(img, None)
        rows.append(dict(image=p.name, key=key, f_exif=f, src=src, f_pred=f_pred))

    df = pd.DataFrame(rows)
    if cache_csv is not None:
        df[["image", "f_pred"]].to_csv(cache_csv, index=False)

    cams = {}
    for key, g in df.groupby("key"):
        if g["f_exif"].notna().any():
            cams[key] = (float(g["f_exif"].dropna().iloc[0]), g["src"].dropna().iloc[0])
        elif g["f_pred"].notna().any():
            cams[key] = (float(g["f_pred"].median()), f"depthpro-median(n={g['f_pred'].notna().sum()})")
        else:
            cams[key] = (None, None)
    return cams, df

### 3.4 Model metric depth

Output berupa **z-depth dalam meter** (jarak sepanjang sumbu optik), bukan jarak lurus ke titik. Karena itu titik 3D = `z · [(u−cx)/f, (v−cy)/f, 1]` dan ray tidak dinormalisasi (beda dengan kode lama).

Depth Pro mengestimasi focal sendiri (`f_pred`), dan depth hasilnya sebanding linear dengan focal yang diasumsikan. Kalau focal kamera sudah ditentukan (3.3), depth dikoreksi dengan faktor `f_known / f_pred` supaya konsisten dengan intrinsik yang dipakai untuk ray. Skala absolutnya tetap dikalibrasi lagi di fit bidang (3.5).

In [ ]:
class MetricDepth:
    def __init__(self, backend="depthpro", device="cuda"):
        self.backend, self.device = backend, device
        if backend == "depthpro":
            from transformers import DepthProImageProcessorFast, DepthProForDepthEstimation
            self.proc = DepthProImageProcessorFast.from_pretrained("apple/DepthPro-hf")
            self.model = DepthProForDepthEstimation.from_pretrained("apple/DepthPro-hf")
        elif backend == "dav2":
            name = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf"
            self.proc = AutoImageProcessor.from_pretrained(name)
            self.model = AutoModelForDepthEstimation.from_pretrained(name)
        else:
            raise ValueError(backend)
        self.model.to(device).eval()

    @torch.no_grad()
    def __call__(self, pil_img, f_known_px: Optional[float] = None):
        W, H = pil_img.size
        inputs = self.proc(images=pil_img, return_tensors="pt").to(self.device)
        out = self.model(**inputs)
        f_pred = None
        if self.backend == "depthpro":
            post = self.proc.post_process_depth_estimation(out, target_sizes=[(H, W)])[0]
            depth = post["predicted_depth"].float().cpu().numpy()
            if post.get("focal_length") is not None:
                f_pred = float(post["focal_length"])
            if f_known_px and f_pred:
                depth = depth * (f_known_px / f_pred)   # selaraskan skala depth dengan focal yang dipakai
        else:
            d = out.predicted_depth.unsqueeze(1)
            depth = nn.functional.interpolate(d, size=(H, W), mode="bicubic",
                                              align_corners=False).squeeze().float().cpu().numpy()
        return np.clip(depth, 0, None), f_pred

### 3.5 Geometri

**Frame kamera** (konvensi OpenCV): x kanan, y bawah, z depan.

- **`fit_ground_plane`**: RANSAC bidang `n·P + d = 0` pada titik 3D trotoar. `n` = normal tanah (arah atas) → pitch/roll kamera sebenarnya; `d` = tinggi kamera hasil fit (`h_fit`); `scale = CAM_HEIGHT_M / h_fit`.
- **`level_frame_from_normal`**: sumbu horizontal depan (`f_h`) dan kanan (`r_h`) dari normal tanah, sehingga pitch/roll terkoreksi tanpa sensor IMU.
- **`to_enu`**: `east = a·cos h + b·sin h`, `north = −a·sin h + b·cos h` (a = kanan, b = depan). Cek: heading 90° → depan = timur, kanan = selatan.
- **`enu_to_latlon`**: bidang singgung lokal dengan jari-jari meridian `M` dan prime vertical `N` elipsoid WGS84; galat bisa diabaikan untuk jarak < 100 m.

In [ ]:
def pixel_rays(us, vs, W, H, f_px):
    # Principal point diasumsikan di tengah gambar (umum untuk kamera HP tanpa kalibrasi)
    us, vs = np.asarray(us, float), np.asarray(vs, float)
    return np.stack([(us - W / 2.0) / f_px, (vs - H / 2.0) / f_px, np.ones_like(us)], axis=-1)


def fit_ground_plane(P, iters=300, thr=0.08, min_inliers=300, seed=0):
    # iters=300: N = log(1-p)/log(1-w^3) ~ 252 untuk p=0.999, w=0.3 (Fischler & Bolles, 1981)
    # thr=0.08 m: kekasaran ubin + noise depth (heuristik, cek rasio inlier di sel 5.4)
    if len(P) < min_inliers:
        return None
    rng = np.random.default_rng(seed)   # seed tetap -> hasil bisa direproduksi
    best_inl = None
    for _ in range(iters):
        a, b, c = P[rng.choice(len(P), 3, replace=False)]
        n = np.cross(b - a, c - a)
        nn_ = np.linalg.norm(n)
        if nn_ < 1e-9:
            continue
        n /= nn_
        d = -n @ a
        inl = np.abs(P @ n + d) < thr
        if best_inl is None or inl.sum() > best_inl.sum():
            best_inl = inl
    if best_inl is None or best_inl.sum() < min_inliers:
        return None
    Q = P[best_inl]
    c = Q.mean(axis=0)
    n = np.linalg.svd(Q - c)[2][-1]     # refinement least-squares pada semua inlier
    d = -n @ c
    if d < 0:                           # normal diarahkan ke sisi kamera (atas)
        n, d = -n, -d
    # tolak kalau bukan "lantai": normal harus dalam 35° dari arah atas kamera (0,-1,0)
    if n @ np.array([0.0, -1.0, 0.0]) < math.cos(math.radians(35)):
        return None
    return n, d, best_inl


def level_frame_from_normal(n_up):
    z = np.array([0.0, 0.0, 1.0])
    f_h = z - (z @ n_up) * n_up
    f_h /= np.linalg.norm(f_h)
    r_h = np.cross(f_h, n_up)
    return f_h, r_h


def level_frame_from_exif(pitch_deg, roll_deg):
    # fallback kalau fit bidang gagal (HP biasanya tidak menyimpan pitch/roll -> dianggap 0)
    p, r = math.radians(pitch_deg), math.radians(roll_deg)
    Rz = np.array([[math.cos(r), -math.sin(r), 0], [math.sin(r), math.cos(r), 0], [0, 0, 1]])
    Rx = np.array([[1, 0, 0], [0, math.cos(p), math.sin(p)], [0, -math.sin(p), math.cos(p)]])
    R = Rx @ Rz
    r_h, down, f_h = R[:, 0], R[:, 1], R[:, 2]
    return f_h, r_h, -down


def to_enu(P, f_h, r_h, heading_deg):
    a, b = P @ r_h, P @ f_h
    h = math.radians(heading_deg)
    return a * math.cos(h) + b * math.sin(h), -a * math.sin(h) + b * math.cos(h)


def enu_to_latlon(lat0, lon0, east_m, north_m):
    a, e2 = 6378137.0, 6.69437999014e-3
    phi = math.radians(lat0)
    s = math.sin(phi)
    M = a * (1 - e2) / (1 - e2 * s * s) ** 1.5
    N = a / math.sqrt(1 - e2 * s * s)
    return (lat0 + math.degrees(north_m / M),
            lon0 + math.degrees(east_m / (N * math.cos(phi))))

### 3.6 Segmentasi dan helper obstacle

- `bottom_on_sidewalk`: obstacle dianggap di trotoar kalau ≥ 30% strip tipis di tepi bawah box masuk mask. Menggantikan cek satu piksel di kode lama, yang bahkan jatuh setengah tinggi box **di bawah** objek (`offset_y_percent=1.0`).
- `contact_depth`: median depth di patch kecil tengah-bawah box, lebih tahan noise daripada satu piksel.

In [ ]:
def sidewalk_mask(seg_model, processor, pil_img, device):
    enc = processor(images=pil_img, return_tensors="pt")
    with torch.no_grad():
        logits = seg_model.model(pixel_values=enc["pixel_values"].to(device)).logits
    logits = nn.functional.interpolate(logits, size=pil_img.size[::-1],
                                       mode="bilinear", align_corners=False)
    return logits.argmax(dim=1).squeeze().cpu().numpy() == 1


def bottom_on_sidewalk(mask, x1, y2, x2, strip_px=6, min_frac=0.3):
    # strip 6 px & 30%: toleran terhadap mask yang tidak rapi di batas objek (heuristik)
    h, w = mask.shape
    xa, xb = int(max(0, x1)), int(min(w, x2))
    ya, yb = int(max(0, y2 - strip_px)), int(min(h, y2 + 1))
    if xb <= xa or yb <= ya:
        return False
    return mask[ya:yb, xa:xb].mean() >= min_frac


def contact_depth(depth, x1, y1, x2, y2):
    # patch tengah 30% lebar, 8% tinggi terbawah box -> menghindari tepi box yang sering berisi latar
    h, w = depth.shape
    bw, bh = x2 - x1, y2 - y1
    xa, xb = int(max(0, x1 + 0.35 * bw)), int(min(w, x2 - 0.35 * bw))
    ya, yb = int(max(0, y2 - 0.08 * bh)), int(min(h, y2 + 1))
    patch = depth[ya:yb, xa:max(xb, xa + 1)]
    return float(np.median(patch)) if patch.size else float("nan")

### 3.7 Proses satu foto

Mengembalikan `rows` (hasil per obstacle) dan `dbg` (data antara untuk visualisasi Bagian 5).

Kolom output: `dist_m` jarak horizontal kamera → obstacle · `diff_m` selisih metode bidang vs depth langsung (indikator kualitas titik kontak, bukan akurasi absolut) · `h_fit`, `scale` tinggi kamera hasil fit dan faktor koreksi · `pitch_fit` pitch hasil fit · `heading` heading geografis yang dipakai.

In [ ]:
def process_image(image_path, seg_model, seg_proc, yolo_model, depth_model, device, cfg, cam_focals,
                  verbose=True):
    log = print if verbose else (lambda *a, **k: None)
    dbg = {}
    pose = read_exif(str(image_path))
    dbg["pose"] = pose
    if pose["lat"] is None or pose["heading"] is None:
        log("  [skip] missing GPS or heading in EXIF")   # tidak di-default ke 0
        return [], dbg
    heading, decl = true_heading(pose, cfg["MAG_DECLINATION_DEG"])
    dbg.update(heading=heading, decl=decl)

    # exif_transpose: hormati EXIF Orientation supaya piksel sesuai yang dilihat kamera
    pil_img = ImageOps.exif_transpose(Image.open(image_path)).convert("RGB")
    W, H = pil_img.size
    key = camera_key(pose, W, H)
    f_known, f_src = cam_focals.get(key, (None, None))      # focal per kamera (3.3)
    depth, f_pred = depth_model(pil_img, f_known)
    f_px = f_known or f_pred                                 # fallback per foto hanya kalau kamera tak dikenal
    if f_known is None and f_pred:
        f_src = "depthpro-single"
    dbg.update(img=pil_img, depth=depth, f_known=f_known, f_pred=f_pred, f_px=f_px, f_src=f_src, camera=key)
    if f_px is None:
        log("  [skip] no focal length")
        return [], dbg
    log(f"  heading: {pose['heading']:.1f} ({pose['heading_ref']}) + decl {decl:.2f} -> {heading:.1f} true | "
        f"focal: f_px={f_px:.0f} ({f_src}), f_pred={f_pred}")

    mask = sidewalk_mask(seg_model, seg_proc, pil_img, device)
    dbg["mask"] = mask

    # --- bidang tanah dari depth piksel trotoar ---
    vs, us = np.nonzero(mask)
    if len(us):
        # subsample 20k piksel: cukup untuk RANSAC, jauh lebih cepat daripada jutaan piksel
        sel = np.random.default_rng(0).choice(len(us), min(len(us), 20000), replace=False)
        us, vs = us[sel], vs[sel]
    z = depth[vs, us]
    ok = (z > cfg["PLANE_MIN_Z"]) & (z < cfg["PLANE_MAX_Z"])
    us, vs, z = us[ok], vs[ok], z[ok]
    P = pixel_rays(us, vs, W, H, f_px) * z[:, None]
    plane = fit_ground_plane(P)

    if plane is not None:
        n_up, h_fit, inl = plane
        scale = cfg["CAM_HEIGHT_M"] / h_fit if cfg["CAM_HEIGHT_M"] else 1.0   # koreksi skala metrik
        f_h, r_h = level_frame_from_normal(n_up)
        pitch_fit = math.degrees(math.asin(np.clip(n_up[2], -1, 1)))
        method = "plane"
        dbg.update(plane_uv=(us, vs), plane_inl=inl)
        log(f"  plane: h_fit={h_fit:.2f} m, scale={scale:.3f}, pitch_fit={pitch_fit:.1f} deg, "
            f"inliers={int(inl.sum())}/{len(inl)}")
    else:
        if cfg["CAM_HEIGHT_M"] is None:
            log("  [skip] plane fit failed and no camera height for fallback")
            return [], dbg
        f_h, r_h, n_up = level_frame_from_exif(pose["pitch"] or 0.0, pose["roll"] or 0.0)
        h_fit, scale, pitch_fit, method = cfg["CAM_HEIGHT_M"], 1.0, None, "level"
        log("  [warn] plane fit failed -> assuming level camera (pitch 0) + CAM_HEIGHT_M")
    dbg.update(h_fit=h_fit, scale=scale, pitch_fit=pitch_fit, method=method)

    res = yolo_model.predict(source=pil_img, verbose=False)[0]
    dbg["all_boxes"] = [(b, float(c), res.names[int(k)]) for b, c, k in
                        zip(res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy(),
                            res.boxes.cls.cpu().numpy().astype(int))]
    rows = []
    for (x1, y1, x2, y2), conf, name in dbg["all_boxes"]:
        if conf < cfg["MIN_CONF"] or not bottom_on_sidewalk(mask, x1, y2, x2):
            continue
        u, v = (x1 + x2) / 2.0, min(y2, H - 1)     # titik kontak = tepi bawah box
        ray = pixel_rays([u], [v], W, H, f_px)[0]

        denom = n_up @ ray
        if denom >= -1e-6:                          # ray di atas horizon -> tidak menyentuh tanah
            continue
        P_plane = ray * (-h_fit / denom) * scale    # posisi utama: ray ∩ bidang

        zc = contact_depth(depth, x1, y1, x2, y2)
        P_depth = ray * zc * scale if np.isfinite(zc) else None   # pembanding

        e, n = to_enu(P_plane, f_h, r_h, heading)
        dist = math.hypot(e, n)
        if dist > cfg["MAX_DIST_M"]:                # error jarak naik ~D² -> buang yang terlalu jauh
            continue
        diff = float(np.linalg.norm(P_plane - P_depth)) if P_depth is not None else float("nan")
        if cfg["USE_DEPTH_POSITION"] and P_depth is not None:
            e, n = to_enu(P_depth, f_h, r_h, heading)
        lat, lon = enu_to_latlon(pose["lat"], pose["lon"], e, n)
        rows.append(dict(image=Path(image_path).name, name=name, conf=conf,
                         method=method, u=u, v=v, x1=x1, y1=y1, x2=x2, y2=y2,
                         dist_m=dist, diff_m=diff, east_m=float(e), north_m=float(n),
                         h_fit=float(h_fit), scale=float(scale), pitch_fit=pitch_fit,
                         heading=heading, lat=lat, lon=lon))
    return rows, dbg

## 4. Load model dan focal per kamera

In [ ]:
seg_model = SegformerFinetuner.load_from_checkpoint(SEGFORMER_CKPT, id2label=ID2LABEL).eval().to(device)
seg_proc = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
seg_proc.do_reduce_labels = False
yolo_model = YOLO(YOLO_WEIGHTS)
depth_model = MetricDepth(cfg["DEPTH_BACKEND"], device)
print("Models loaded")

Focal ditentukan sekali per kamera sebelum proses. Untuk kamera tanpa focal valid di EXIF (seperti OPPO A7), Depth Pro dijalankan sekali per foto untuk estimasi focal, lalu diambil mediannya. Hasil per foto disimpan di `focal_cache.csv`, jadi run berikutnya tidak menghitung ulang.

In [ ]:
image_list = sorted(p for p in IMAGE_FOLDER.iterdir() if p.suffix.lower() in (".jpg", ".jpeg"))
print(len(image_list), "images")
CAM_FOCALS, focal_table = estimate_camera_focals(image_list, depth_model, cfg,
                                                 cache_csv=IMAGE_FOLDER / "focal_cache.csv")
for k, (f, src) in CAM_FOCALS.items():
    print(f"{k}  ->  f_px = {f:.1f}  ({src})" if f else f"{k}  ->  focal tidak diketahui")
display(focal_table)

## 5. Debug satu foto

Jalankan untuk beberapa foto sebelum batch. Yang dicek:
- **5.1 Pose**: heading ada? `heading_ref` = `M`? focal dari sumber apa?
- **5.2 Mask trotoar**: apakah trotoar tertangkap dengan benar?
- **5.3 Depth**: trotoar di bagian bawah foto seharusnya sekitar 2–5 m
- **5.4 Inlier bidang**: titik hijau seharusnya menutupi permukaan trotoar
- **5.5 Deteksi**: hijau = lolos filter, abu-abu = dibuang
- **5.6 Tampak atas**: posisi obstacle relatif terhadap kamera (E/N, meter)

In [ ]:
IDX = 0   # ganti untuk cek foto lain
test_path = image_list[IDX]
print(test_path.name)
rows, dbg = process_image(test_path, seg_model, seg_proc, yolo_model, depth_model, device, cfg, CAM_FOCALS)

### 5.1 Pose dan focal

In [ ]:
info = dict(dbg["pose"], declination=dbg.get("decl"), heading_true=dbg.get("heading"),
            camera=dbg.get("camera"), f_px=dbg.get("f_px"), f_src=dbg.get("f_src"), f_pred=dbg.get("f_pred"))
pd.Series(info).to_frame("value")

### 5.2 Mask trotoar

In [ ]:
img = np.array(dbg["img"])
mask = dbg["mask"]
overlay = img.copy()
overlay[mask] = (0.5 * overlay[mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
plt.figure(figsize=(12, 6)); plt.imshow(overlay); plt.title("Sidewalk mask (hijau)"); plt.axis("off"); plt.show()

### 5.3 Depth map (meter)

In [ ]:
depth = dbg["depth"]
plt.figure(figsize=(12, 6))
im = plt.imshow(depth, cmap="turbo", vmin=0, vmax=np.percentile(depth, 95))
plt.colorbar(im, label="depth (m), sebelum koreksi skala")
plt.title(f"Metric depth ({cfg['DEPTH_BACKEND']})"); plt.axis("off"); plt.show()
print(f"Median depth di trotoar: {np.median(depth[mask]):.2f} m" if mask.any() else "No sidewalk pixels")

### 5.4 Inlier bidang tanah

In [ ]:
if "plane_uv" in dbg:
    us, vs = dbg["plane_uv"]; inl = dbg["plane_inl"]
    plt.figure(figsize=(12, 6)); plt.imshow(img)
    plt.scatter(us[~inl], vs[~inl], s=1, c="red", label="outlier")
    plt.scatter(us[inl], vs[inl], s=1, c="lime", label="inlier")
    plt.legend(markerscale=10); plt.axis("off")
    plt.title(f"Ground plane: h_fit={dbg['h_fit']:.2f} m, scale={dbg['scale']:.3f}, "
              f"pitch_fit={dbg['pitch_fit']:.1f}°, inlier {inl.mean():.0%}")
    plt.show()
else:
    print("Plane fit gagal, pakai fallback kamera datar + CAM_HEIGHT_M")

### 5.5 Deteksi obstacle

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6)); ax.imshow(img)
kept = {(r["x1"], r["y1"], r["x2"], r["y2"]): r for r in rows}
for (x1, y1, x2, y2), conf, name in dbg.get("all_boxes", []):
    r = kept.get((x1, y1, x2, y2))
    color = "lime" if r else "gray"
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=2))
    label = f"{name} {conf:.2f}"
    if r:
        label += f"\n{r['dist_m']:.1f} m, diff {r['diff_m']:.1f} m"
        ax.plot(r["u"], r["v"], "r+", ms=12, mew=2)
    ax.text(x1, y1 - 4, label, color=color, fontsize=8, va="bottom",
            bbox=dict(fc="black", alpha=0.5, pad=1))
ax.axis("off"); ax.set_title("Hijau = lolos filter, abu-abu = dibuang, + = titik kontak"); plt.show()
pd.DataFrame(rows)[["name", "conf", "dist_m", "diff_m", "east_m", "north_m", "lat", "lon"]] if rows else "No obstacles"

### 5.6 Tampak atas (relatif terhadap kamera)

In [ ]:
if rows:
    df1 = pd.DataFrame(rows)
    h = math.radians(dbg["heading"])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df1.east_m, df1.north_m, c="red")
    for _, r in df1.iterrows():
        ax.annotate(r["name"], (r.east_m, r.north_m), fontsize=8)
    ax.plot(0, 0, "k^", ms=10, label="kamera")
    ax.annotate("", xy=(3 * math.sin(h), 3 * math.cos(h)), xytext=(0, 0),   # panah tipis 3 m: tidak menutupi titik
                arrowprops=dict(arrowstyle="->", color="k", lw=1.5))
    ax.set_xlabel("East (m)"); ax.set_ylabel("North (m)"); ax.set_aspect("equal"); ax.grid(True)
    ax.set_title(f"Heading {dbg['heading']:.1f}° (true)"); ax.legend(); plt.show()

## 6. Batch seluruh folder

Seperti kode asli: satu `.shp` per foto di samping file fotonya, lalu digabung ke `results_all.shp`, ditambah `cameras.shp` (posisi, heading, dan kamera tiap foto). Nama kolom ≤ 10 karakter supaya aman di Shapefile.

In [ ]:
def export_shp(df, shp_path):
    gdf = gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df["lon"], df["lat"])], crs="EPSG:4326")
    gdf.to_file(shp_path, driver="ESRI Shapefile")
    return gdf

all_dfs, summary = [], []
for p in image_list:
    print(f"\n=== {p.name} ===")
    r, d = process_image(p, seg_model, seg_proc, yolo_model, depth_model, device, cfg, CAM_FOCALS)
    print(f"  {len(r)} obstacle(s)")
    pose = d["pose"]
    summary.append(dict(image=p.name, camera=d.get("camera"), lat=pose.get("lat"), lon=pose.get("lon"),
                        head_mag=pose.get("heading"), decl=d.get("decl"), heading=d.get("heading"),
                        method=d.get("method"), h_fit=d.get("h_fit"), scale=d.get("scale"),
                        pitch_fit=d.get("pitch_fit"), f_px=d.get("f_px"), f_src=d.get("f_src"),
                        f_pred=d.get("f_pred"), n_obs=len(r)))
    if r:
        df = pd.DataFrame(r)
        export_shp(df, p.with_suffix(".shp"))
        all_dfs.append(df)

summary_df = pd.DataFrame(summary)
summary_df

### 6.1 Ringkasan kualitas

- `scale`: koreksi skala Depth Pro per foto. Variasi besar antar foto (uji awal 1.05–1.84) menunjukkan skala metrik model tidak stabil, sehingga `CAM_HEIGHT_M` menentukan.
- `pitch_fit`: sudut tunduk HP; konsisten antar foto menandakan fit bidang stabil.
- `diff_m`: selisih metode bidang vs depth. Konsistensi internal, bukan akurasi absolut; nilai besar menandai titik kontak bermasalah.
- `f_pred` per foto vs `f_px` per kamera: menunjukkan seberapa jauh estimasi focal per foto bervariasi.

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    summary_df["scale"].dropna().plot.hist(ax=axes[0], bins=20, title="scale (CAM_HEIGHT / h_fit)")
    summary_df["pitch_fit"].dropna().plot.hist(ax=axes[1], bins=20, title="pitch_fit (deg)")
    if all_dfs:
        pd.concat(all_dfs)["diff_m"].dropna().plot.hist(ax=axes[2], bins=20, title="diff_m plane vs depth (m)")
    plt.tight_layout(); plt.show()

    display(summary_df[["image", "camera", "scale", "h_fit", "pitch_fit", "f_px", "f_src", "f_pred", "n_obs"]])
    if all_dfs:
        obs = pd.concat(all_dfs, ignore_index=True)
        display(obs.sort_values("diff_m", ascending=False)[["image", "name", "conf", "dist_m", "diff_m"]].head(5))

    # Cek silang focal EXIF/kalibrasi vs estimasi Depth Pro per kamera.
    # Beda > 15% -> kemungkinan EXIF focal kamera itu salah; pertimbangkan kalibrasi (CAMERA_CALIB).
    chk = summary_df.dropna(subset=["f_px", "f_pred"]).assign(ratio=lambda d: d.f_pred / d.f_px)
    if not chk.empty:
        focal_check = chk.groupby(["camera", "f_src"])["ratio"].agg(["median", "count"])
        display(focal_check)
        for (cam, src), r in focal_check.iterrows():
            if not src.startswith("depthpro") and abs(r["median"] - 1) > 0.15:
                print(f"[warn] {cam}: focal {src} beda {abs(r['median'] - 1):.0%} dari estimasi Depth Pro")

### 6.2 Gabung jadi satu shapefile + posisi kamera

In [ ]:
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_shp = IMAGE_FOLDER / "results_all.shp"
    export_shp(final_df, final_shp)
    print(f"Semua hasil digabung: {final_shp} ({len(final_df)} titik)")
else:
    print("Tidak ada obstacle yang bisa disimpan.")

cams = summary_df.dropna(subset=["lat", "lon"])
if not cams.empty:
    export_shp(cams, IMAGE_FOLDER / "cameras.shp")
    print(f"Posisi kamera: {IMAGE_FOLDER / 'cameras.shp'} ({len(cams)} foto)")

## 7. Cek hasil di QGIS

Contoh hasil ada di `sample_data/results/` di repo. Download `results_all` dan `cameras`, masing-masing **semua** file pendampingnya (`.shp .shx .dbf .prj .cpg`) dalam satu folder.

1. **Basemap**: Browser → XYZ Tiles → OpenStreetMap. Untuk citra satelit: New Connection, URL `https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}`, max zoom 19.
2. **Panah heading**: `cameras` → Symbology → Simple Marker bentuk arrow → Rotation data-defined = `heading`.
3. **Garis kamera → obstacle**: Processing → *Join by lines (hub lines)*; hub = `cameras` (field `image`), spoke = `results_all` (field `image`).
4. **Warna kualitas**: `results_all` → Rule-based: `"diff_m" <= 1` hijau, selain itu merah.
5. **Ukur jarak**: Measure Line (`Ctrl+Shift+M`), mode Ellipsoidal, satuan meter.

Yang dicek: kamera berada di trotoar tempat memotret · panah sejajar arah jalan seperti di foto · obstacle (mis. bollard) berderet di tepi trotoar.

## Catatan sumber error

- **Posisi GPS kamera** menggeser semua obstacle dari foto yang sama secara bersamaan. Kanopi pohon melemahkan dan memantulkan sinyal GNSS. **Uji awal:** foto di tempat terbuka (IMG_20250905_172550) jatuh tepat di trotoar, sedangkan dua foto di bawah pohon bergeser.
- **Heading kompas HP**: error 5° di jarak 15 m ≈ 1.3 m ke samping.
- **`CAM_HEIGHT_M`**: menentukan skala; ukur, jangan ditebak.
- **Focal**: kalibrasi checkerboard (OpenCV `calibrateCamera`) per kamera lebih akurat daripada estimasi Depth Pro; isi hasilnya di `CAMERA_CALIB`.